### Medallion Architecture: Bronze, Silver, Gold

**Bronze Layer:**
* Ingests raw data from the source table (`workspace.ecommerce.events`)
* Adds an ingestion timestamp for traceability
* Stores all source columns in a managed Delta table (`workspace.ecommerce.bronze_events`)

**Silver Layer:**
* Cleans and validates Bronze data
* Removes duplicates and rows with missing key fields
* Enforces schema consistency
* Writes cleaned data to a managed Delta table (`workspace.ecommerce.silver_events`)

**Gold Layer:**
* Aggregates and summarizes Silver data for business analytics
* Calculates metrics (e.g., daily sales, event counts, average price)
* Stores results in a managed Delta table (`workspace.ecommerce.gold_events`)

---
#### Practical Notes & Recommendations
* Managed Delta tables ensure reliability, ACID compliance, and incremental processing
* Bronze layer preserves raw data for audit and traceability
* Silver layer enforces data quality; extend with custom validation as needed
* Gold layer powers dashboards and analytics; customize metrics and groupings for your use case
* For production, automate each layer with jobs/workflows and add error logging
* Incremental processing can be enhanced with watermarks or change data capture
* This structure provides a robust foundation for scalable analytics in Databricks


In [0]:
from pyspark.sql.functions import current_timestamp

# Step 1: Read raw data from source table
events_df = spark.read.table("workspace.ecommerce.events")

# Step 2: Add ingestion timestamp
bronze_df = events_df.withColumn("ingestion_ts", current_timestamp())

# Step 3: Write to Bronze Delta table (append mode)
bronze_table = "workspace.ecommerce.bronze_events"
bronze_df.write.format("delta").mode("append").saveAsTable(bronze_table)

# Step 4: Display sample Bronze data
display(spark.read.table(bronze_table).limit(10))

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Step 1: Read raw data from Bronze table
bronze_table = "workspace.ecommerce.bronze_events"
bronze_df = spark.read.table(bronze_table)

# Step 2: Data cleaning and validation
# Remove duplicates based on key columns
key_cols = ["event_time", "event_type", "product_id", "user_id", "user_session"]
silver_df = bronze_df.dropDuplicates(key_cols)

# Handle missing values (example: drop rows with null product_id or user_id)
silver_df = silver_df.dropna(subset=["product_id", "user_id"])

# Step 3: Write to Silver Delta table (overwrite for initial run)
silver_table = "workspace.ecommerce.silver_events"
silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

# Step 4: Display sample Silver data
display(spark.read.table(silver_table).limit(10))


In [0]:
from pyspark.sql.functions import sum, count, avg, to_date

# Step 1: Read cleaned data from Silver table
silver_table = "workspace.ecommerce.silver_events"
silver_df = spark.read.table(silver_table)

# Step 2: Business aggregation example
# Aggregate daily sales and event counts by product and event type
gold_df = (
    silver_df
    .withColumn("event_date", to_date("event_time"))
    .groupBy("event_date", "product_id", "event_type")
    .agg(
        sum("price").alias("total_sales"),
        count("event_type").alias("event_count"),
        avg("price").alias("avg_price")
    )
)

# Step 3: Write to Gold Delta table (overwrite for initial run)
gold_table = "workspace.ecommerce.gold_events"
gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)

# Step 4: Display sample Gold data
display(spark.read.table(gold_table).limit(10))